# F1 - PyTorch Refresher (Core)
## Building a Barclays Customer Support Intelligence System

You are on the AI team at Barclays. The customer support department receives thousands
of complaint texts every day - account issues, fraud alerts, payment failures, loan
queries. Your job is to build a system that classifies each complaint automatically
so it reaches the right team.

In this notebook you build the foundation: the PyTorch skills every component of
that system depends on.

### What you will build
1. Tensors - how complaint text becomes numbers
2. Dataset and DataLoader - how we feed complaint batches efficiently
3. Classifier with nn.Module - the model that learns to route complaints

This is the lean core path. Two further deep-dive sections - the manual autograd
training loop and the HuggingFace Trainer - now live in the optional companion
notebook `pytorch_optional_deep_dive.ipynb`. Do that one if you finish early or
want the internals; it is not required for the fine-tuning topics.


In [ ]:
# Disable TensorFlow backend in transformers (SageMaker image compatibility).
# Must run before any transformers import.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


In [ ]:
# Environment setup - runs in SageMaker Studio JupyterLab kernel
# No remote training in this notebook

import subprocess, sys

# Pin numpy<2 to avoid compatibility issues with older torch ops
# transformers and datasets needed for Section 5 (HuggingFace Trainer)
# sagemaker pinned to 2.x (3.x has breaking changes)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy<2",
    "sagemaker>=2.200.0,<3.0.0",
    "transformers>=4.35.0,<4.40.0",
    "tokenizers>=0.15.0,<0.20.0",
    "datasets>=2.18.0,<3.0.0",
], check=True)

import sagemaker
from sagemaker import get_execution_role
import boto3

sess = sagemaker.Session()
role = get_execution_role()
bucket = sess.default_bucket()
region = sess.boto_region_name

print(f"Role: {role}")
print(f"Bucket: {bucket}")
print(f"Region: {region}")
print("Environment OK - running in Studio kernel, no remote training needed for F1")

In [ ]:
import torch as pt
import torch.nn as nn
import numpy as np
import random
import matplotlib
matplotlib.use("Agg")          # SageMaker Studio: use non-interactive backend
import matplotlib.pyplot as plt

# Reproducibility seed
SEED = 42
pt.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f"PyTorch version: {pt.__version__}")
print(f"CUDA available:  {pt.cuda.is_available()}")
device = pt.device("cuda" if pt.cuda.is_available() else "cpu")
print(f"Device:          {device}")

## Section 1 - Tensors: How Complaint Text Becomes Numbers

Customer complaint text must become numbers before any model can process it.
Those numbers live in **tensors** -- PyTorch's core data structure, GPU-ready and differentiable.

In [ ]:
# --- Beat 1: The naive approach - why plain Python lists fail ---
#
# Suppose our customer complaints look like this (bag-of-words encoding):
# [word_count_account, word_count_fraud, word_count_payment, word_count_loan]

complaints_list = [
    [3, 0, 1, 0],   # complaint A: "account issue account payment"
    [0, 5, 0, 1],   # complaint B: "fraud fraud fraud fraud fraud loan"
    [1, 0, 4, 0],   # complaint C: "payment payment account payment payment"
]

labels_list = [0, 1, 0]   # 0=account_issue, 1=fraud, 0=account_issue

# Try to compute a dot product between complaint vectors and a weight vector
weights_list = [0.5, 2.0, 0.3, 1.0]

try:
    # This is what we WANT: matrix multiply complaints by weights
    result = complaints_list * weights_list   # <-- wrong: list * list = ERROR
    print(result)
except TypeError as e:
    print(f"ERROR: {e}")

# Even if we use a loop, it is SLOW and has no gradients
dot_products = []
for complaint in complaints_list:
    dot = sum(c * w for c, w in zip(complaint, weights_list))
    dot_products.append(dot)

print(f"\nDot products (manual loop): {dot_products}")
print("Problem: this is slow for 100,000 complaints and PyTorch cannot")
print("compute gradients through plain Python loops.")

<!-- DIAGRAM: tensor-shapes-and-matmul -->

```mermaid
graph LR
    A["complaints<br/>shape [3, 4]<br/>3 complaints x 4 features"] -->|"matmul @"| C["scores<br/>shape [3]<br/>one score per complaint"]
    B["weights<br/>shape [4]<br/>one weight per feature"] -->|"matmul @"| C
    C --> D["CPU or GPU<br/>via .to(device)"]

    style A fill:#ddf,stroke:#33c
    style B fill:#ddf,stroke:#33c
    style C fill:#dfd,stroke:#3a3
    style D fill:#ffd,stroke:#cc3
```

*Figure: A single matmul replaces a Python loop over every complaint. Tensors carry shape information so PyTorch can dispatch the operation to optimized BLAS kernels on CPU or CUDA kernels on GPU - the same code runs on both.*

The diagram shows how one matrix multiply replaces a Python loop over every
complaint. Tensors carry shape information so PyTorch can dispatch the operation
to optimized CPU or GPU kernels - the same code runs on both with a single
`.to(device)` call.

In [ ]:
# --- Beat 3: Tensors solve all three problems ---
# Fast, differentiable, GPU-ready

# 1. Create tensors from the same complaint data
complaints = pt.tensor([
    [3, 0, 1, 0],
    [0, 5, 0, 1],
    [1, 0, 4, 0],
], dtype=pt.float32)    # float32 is the standard dtype for neural nets

labels = pt.tensor([0, 1, 0], dtype=pt.long)  # long = int64, required by CrossEntropyLoss

weights = pt.tensor([0.5, 2.0, 0.3, 1.0], dtype=pt.float32)

print("complaints shape:", complaints.shape)   # [3, 4]: 3 complaints, 4 features
print("labels shape:    ", labels.shape)       # [3]
print("weights shape:   ", weights.shape)      # [4]

# 2. Matrix multiply: all 3 complaints x weights in ONE call
scores = complaints @ weights    # shape: [3]
print("\nScores (matmul):", scores)

# 3. Common creation patterns
zeros = pt.zeros(3, 4)           # 3x4 block of zeros
ones  = pt.ones(3, 4)            # 3x4 block of ones
rand  = pt.randn(3, 4)           # 3x4 from standard normal
arange = pt.arange(0, 10, 2)    # [0, 2, 4, 6, 8]
linspace = pt.linspace(0, 1, 5) # [0.0, 0.25, 0.5, 0.75, 1.0]

print("\nzeros:\n", zeros)
print("ones:\n",  ones)
print("arange:",  arange)
print("linspace:", linspace)

# 4. Indexing and slicing (same as NumPy)
print("\nFirst complaint:", complaints[0])          # row 0
print("Feature 1 for all complaints:", complaints[:, 1])  # column 1
print("Top-left 2x2:", complaints[:2, :2])

# 5. Reductions
print("\nMax score:", scores.max().item())
print("Mean complaint vector:", complaints.mean(dim=0))  # mean per feature

# 6. Reshape
flat = complaints.view(-1)          # flatten to 1D: [12]
back = flat.view(3, 4)              # back to original shape
print("\nFlattened shape:", flat.shape)
print("Reshaped shape:", back.shape)

# 7. GPU move (works even if no GPU - device is "cpu" then)
complaints_on_device = complaints.to(device)
print(f"\ncomplaints device: {complaints_on_device.device}")

### Lab 1 - Tensor Foundations (Tier 1, ~15 min)

**Situation**: You have received a batch of 200 Barclays customer complaints.
Each complaint has been encoded into 6 features (bag-of-words counts for:
account, fraud, payment, loan, dispute, refund).

**Task**: Create the complaint feature matrix and label tensor, then explore
the data structure the model will train on.

**Action**: Complete the steps below. Each `= None  # YOUR CODE` is one step.

**Result**: The verification cell at the end will confirm your shapes are correct.

Steps:
1. Create a float32 tensor named `X_complaints` with shape [200, 6] using `pt.randn`
   (we use random numbers to simulate the encoded complaint data)
2. Create a long tensor named `y_complaints` with shape [200] containing random class
   labels 0, 1, or 2 (use `pt.randint`)
3. Move both tensors to `device`
4. Compute `feature_means`: the mean value of each of the 6 features across all 200
   complaints (shape should be [6])
5. Compute `class_counts`: how many complaints fall into each class (use `pt.bincount`)

**Stretch**: Normalize `X_complaints` so each feature has mean 0 and std 1 across
all 200 samples (hint: subtract mean, divide by std, using `dim=0`).

In [ ]:
# Lab 1 - Tensor Foundations (SOLUTION)
pt.manual_seed(SEED)

# Step 1: pt.randn generates samples from standard normal distribution
# dtype=pt.float32 is required by neural network layers
X_complaints = pt.randn(200, 6, dtype=pt.float32)

# Step 2: pt.randint(low, high, size) generates integers in [low, high)
# dtype=pt.long = int64, which CrossEntropyLoss requires for labels
y_complaints = pt.randint(0, 3, (200,), dtype=pt.long)

# Step 3: .to(device) moves the tensor to cpu or cuda, whichever is available
X_complaints = X_complaints.to(device)
y_complaints = y_complaints.to(device)

# Step 4: dim=0 reduces along the sample axis, keeping the feature dimension [6]
feature_means = X_complaints.mean(dim=0)

# Step 5: pt.bincount counts occurrences of each integer value 0, 1, 2
class_counts = pt.bincount(y_complaints)

print("X_complaints shape:", X_complaints.shape)
print("y_complaints shape:", y_complaints.shape)
print("feature_means:", feature_means)
print("class_counts:", class_counts)

In [ ]:
# Lab 1 safety-net: run this if you did not finish Lab 1.
# SKIP this cell if you DID finish Lab 1.
pt.manual_seed(SEED)
if X_complaints is None:
    print("Using Lab 1 safety-net so the rest of the notebook can run.")
    X_complaints = pt.randn(200, 6, dtype=pt.float32).to(device)
    y_complaints = pt.randint(0, 3, (200,), dtype=pt.long).to(device)
    feature_means = X_complaints.mean(dim=0)
    class_counts = pt.bincount(y_complaints)

In [ ]:
# Verification - Lab 1
assert X_complaints is not None, "X_complaints not created"
assert X_complaints.shape == (200, 6), f"Expected (200,6), got {X_complaints.shape}"
assert X_complaints.dtype == pt.float32, "X_complaints must be float32"
assert y_complaints.shape == (200,), f"Expected (200,), got {y_complaints.shape}"
assert y_complaints.dtype == pt.long, "y_complaints must be long (int64)"
assert str(X_complaints.device).startswith(str(device).split(":")[0]), \
    f"X_complaints not on {device}"
assert feature_means.shape == (6,), f"feature_means shape wrong: {feature_means.shape}"
assert class_counts.shape == (3,), f"class_counts shape wrong: {class_counts.shape}"
print("Lab 1 passed. Shapes and dtypes are correct.")
print(f"  X_complaints: {X_complaints.shape} on {X_complaints.device}")
print(f"  feature_means: {feature_means}")
print(f"  class_counts per class: {class_counts.tolist()}")

**Homework Extension 1 - Tensor Operations**

1. Load the Yelp Polarity dataset from HuggingFace datasets (it is pre-encoded)
   and convert the first 1000 reviews into a bag-of-words tensor using a vocabulary
   of the 50 most common words. Hint: `datasets.load_dataset("yelp_polarity")`.
2. Compute the cosine similarity between every pair of complaint vectors in
   `X_complaints`. What is the maximum similarity? What does a high cosine similarity
   between two complaint vectors mean for the classification task?
3. (Challenge) Implement your own one-hot encoding function that takes a 1D integer
   tensor of class indices and returns a 2D float tensor of one-hot rows, without
   using any loop or list comprehension - pure tensor operations only.

## Section 2 - Dataset and DataLoader: Feeding Complaint Batches Efficiently

Barclays complaint data has millions of records -- loading all of them into one tensor
exhausts memory. `Dataset` describes your data; `DataLoader` handles shuffling,
batching, and multi-worker loading automatically.


In [ ]:
# --- Beat 1: Manual batching is error-prone ---
pt.manual_seed(SEED)

# 200 complaints, 6 features each
X_full = pt.randn(200, 6)
y_full = pt.randint(0, 3, (200,))

BATCH_SIZE = 32

# Attempt to iterate manually
print("Manual batch loop:")
for i in range(0, len(X_full), BATCH_SIZE):
    X_batch = X_full[i : i + BATCH_SIZE]
    y_batch = y_full[i : i + BATCH_SIZE]
    # Problem 1: last batch may be smaller - model might not handle it
    # Problem 2: data is NOT shuffled - model sees same order every epoch
    # Problem 3: no parallel loading - slow for data on disk
    if i == 0:
        print(f"  batch 0 shape: {X_batch.shape}")

# The last batch
last_start = (len(X_full) // BATCH_SIZE) * BATCH_SIZE
X_last = X_full[last_start:]
print(f"  last batch shape: {X_last.shape}")  # might be < BATCH_SIZE
print("\nProblems: no shuffle, variable last-batch size, no parallel loading.")
print("PyTorch DataLoader solves all three.")

In [ ]:
# --- Beat 3: Dataset and DataLoader ---
from torch.utils.data import Dataset, DataLoader, TensorDataset

# 1. TensorDataset: the simplest Dataset - wraps existing tensors
pt.manual_seed(SEED)
X_full = pt.randn(200, 6, dtype=pt.float32).to(device)
y_full = pt.randint(0, 3, (200,), dtype=pt.long).to(device)

dataset = TensorDataset(X_full, y_full)
print(f"Dataset length: {len(dataset)}")
print(f"First sample X shape: {dataset[0][0].shape}")
print(f"First sample y:       {dataset[0][1]}")

# 2. DataLoader: handles batching, shuffling, drop_last
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,     # shuffle at the start of EACH epoch
    drop_last=False,  # keep the last partial batch
)
print(f"\nNumber of batches per epoch: {len(loader)}")   # ceil(200/32) = 7

# 3. Iterate over one epoch
print("\nBatch shapes in one epoch:")
for batch_idx, (X_batch, y_batch) in enumerate(loader):
    if batch_idx < 3 or batch_idx == len(loader) - 1:
        print(f"  batch {batch_idx}: X={X_batch.shape} y={y_batch.shape}")
    elif batch_idx == 3:
        print("  ...")

# 4. Custom Dataset for more complex data (e.g. text that needs tokenization)
class ComplaintDataset(Dataset):
    """A Dataset for Barclays complaint records.

    In production this would tokenize raw text. Here we use pre-computed
    feature vectors to keep the example self-contained.
    """
    def __init__(self, features, labels):
        # Store data - no processing at init time
        self.X = features.to(pt.float32)
        self.y = labels.to(pt.long)

    def __len__(self):
        # DataLoader calls this to know how many samples exist
        return len(self.y)

    def __getitem__(self, idx):
        # DataLoader calls this for each index in a batch
        return self.X[idx], self.y[idx]

# Same data, but now in our custom Dataset
complaint_ds = ComplaintDataset(X_full, y_full)
complaint_loader = DataLoader(complaint_ds, batch_size=32, shuffle=True)
X_b, y_b = next(iter(complaint_loader))
print(f"\nCustom Dataset - first batch: X={X_b.shape} y={y_b.shape}")

### Lab 3 - DataLoader for Complaint Batches (Tier 1, ~15 min)

**Situation**: Your team has a synthetic dataset of 500 Barclays complaints with
8 features each, in 4 complaint categories (account, fraud, payment, general).

**Task**: Build a `ComplaintDataset`, wrap it in a `DataLoader`, and run one full
epoch of a dummy forward pass to confirm shapes are correct.

**Action**: Complete the steps marked `# YOUR CODE`.

**Result**: The verification cell counts the total samples seen across all batches
and confirms it equals 500.

Steps:
1. Create `X_500` (shape [500, 8], float32) and `y_500` (shape [500], long, values 0-3)
2. Implement the `ComplaintDataset.__getitem__` method (one line)
3. Create `train_loader` with `batch_size=64` and `shuffle=True`
4. Iterate over `train_loader` for one epoch, accumulating `total_seen` (count of samples)

**Stretch**: Add a `transform` parameter to `ComplaintDataset.__init__` that accepts
a callable. When `transform` is not None, apply it to `self.X[idx]` before returning.
Test it by passing a lambda that normalizes each row to unit norm.

In [ ]:
# Lab 3 - DataLoader for Complaint Batches (SOLUTION)
pt.manual_seed(SEED)

# Step 1: 500 complaints, 8 features each, 4 classes
X_500 = pt.randn(500, 8, dtype=pt.float32)
# Values 0-3 for 4 classes (account, fraud, payment, general)
y_500 = pt.randint(0, 4, (500,), dtype=pt.long)

class ComplaintDataset(Dataset):
    def __init__(self, features, labels):
        # .to(pt.float32) and .to(pt.long) ensure correct dtypes regardless of input
        self.X = features.to(pt.float32)
        self.y = labels.to(pt.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        # Return a (features, label) tuple for this single sample
        # DataLoader will stack these into batches automatically
        return self.X[idx], self.y[idx]

# Step 3: Wrap in a DataLoader - shuffle=True randomizes order each epoch
_ds = ComplaintDataset(X_500, y_500)
train_loader = DataLoader(_ds, batch_size=64, shuffle=True)

# Step 4: One epoch - count total samples seen
total_seen = 0
for X_batch, y_batch in train_loader:
    total_seen += len(X_batch)   # accumulate sample count

print(f"Total samples seen in one epoch: {total_seen}")
print(f"Expected: 500")

In [ ]:
# Lab 3 safety-net: run this if you did not finish Lab 3.
# SKIP this cell if you DID finish Lab 3.
pt.manual_seed(SEED)
if X_500 is None or train_loader is None:
    print("Using Lab 3 safety-net so the rest of the notebook can run.")
    X_500 = pt.randn(500, 8, dtype=pt.float32)
    y_500 = pt.randint(0, 4, (500,), dtype=pt.long)

    class _ComplaintDataset(Dataset):
        def __init__(self, features, labels):
            self.X = features.to(pt.float32)
            self.y = labels.to(pt.long)
        def __len__(self):
            return len(self.y)
        def __getitem__(self, idx):
            return self.X[idx], self.y[idx]

    _ds = _ComplaintDataset(X_500, y_500)
    train_loader = DataLoader(_ds, batch_size=64, shuffle=True)
    total_seen = sum(len(yb) for _, yb in train_loader)

In [ ]:
# Verification - Lab 3
assert total_seen == 500, f"Expected 500 samples, got {total_seen}"
X_b, y_b = next(iter(train_loader))
assert X_b.shape[1] == 8, f"Expected 8 features, got {X_b.shape[1]}"
assert y_b.dtype == pt.long, "y_b must be long dtype"
print(f"Lab 3 passed. {total_seen} samples seen across {len(train_loader)} batches.")
print(f"Batch shape: X={X_b.shape}, y={y_b.shape}")

**Homework Extension 3 - DataLoader**

1. Implement a `WeightedComplaintSampler` using `torch.utils.data.WeightedRandomSampler`.
   Fraud complaints (class 1) are rare in real data - only 5% of the dataset.
   Make the sampler draw class 1 samples 10x more often to handle class imbalance.
2. Measure the throughput difference (samples/second) between `num_workers=0` and
   `num_workers=4` for a DataLoader with a dataset of 10,000 samples. Note: in
   SageMaker Studio this may not show a large difference since data is in-memory.
3. (Challenge) Build a `TextComplaintDataset` that reads raw complaint strings from
   a Python list, tokenizes them character-by-character (vocabulary = ASCII printable
   chars), pads all sequences to the same length, and returns `(char_ids_tensor, label)`.

## Section 3 - Complaint Classifier with nn.Module

Tensors and data loading are in place. Now we wire them into a real classifier.
Subclass `nn.Module`, define layers in `__init__`, implement `forward()` --
PyTorch handles parameter registration, device placement, and gradient tracking.

**How training works.** A model starts with random weights. Each step it makes a
prediction, the loss measures how wrong it is, and `loss.backward()` computes -
for every weight - which direction reduces the loss. `optimizer.step()` nudges the
weights that way, and `optimizer.zero_grad()` clears the gradients before the next
step. You will see those three calls in the training loop below. The optional
companion notebook opens up exactly what `.backward()` is doing under the hood.


In [ ]:
# --- Beat 1: Building a classifier with raw tensors (do not do this) ---
pt.manual_seed(SEED)

# A two-layer "network" as plain tensors
W1 = pt.randn(6, 16, requires_grad=True)    # layer 1 weights
b1 = pt.zeros(16, requires_grad=True)        # layer 1 bias
W2 = pt.randn(16, 3, requires_grad=True)    # layer 2 weights
b2 = pt.zeros(3, requires_grad=True)         # layer 2 bias

# Forward pass
def bad_forward(x):
    h = pt.relu(x @ W1 + b1)
    return h @ W2 + b2

X_sample = pt.randn(10, 6)
out = bad_forward(X_sample)
print(f"Output shape: {out.shape}")   # works so far

# Problem 1: To save the model, we would need to manually list ALL tensors
# params_to_save = [W1, b1, W2, b2]   # what if we had 20 layers?

# Problem 2: Moving to GPU requires updating every tensor by hand
# W1 = W1.to("cuda"), b1 = b1.to("cuda"), W2 = W2.to("cuda"), ...

# Problem 3: No standardized interface - no model.eval(), model.train(), model.parameters()

try:
    # This is how you WANT to move a model to GPU - but it doesn't work here
    bad_forward.to(device)
except AttributeError as e:
    print(f"\nERROR: {e}")

print("\nWe need nn.Module so PyTorch can manage parameters, device placement,")
print("and train/eval mode automatically.")

In [ ]:
# --- Beat 3: Complaint classifier with nn.Module ---
from torch import optim

class ComplaintClassifier(nn.Module):
    """Two-layer feedforward classifier for Barclays complaint categories.

    Input:  6-dimensional complaint feature vector
    Output: 3-class logits (account_issue=0, fraud=1, payment=2)
    """
    def __init__(self, input_dim=6, hidden_dim=16, num_classes=3):
        super().__init__()
        # nn.Linear registers weights and bias as Parameters automatically
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(hidden_dim, num_classes)
        # No need to list them separately - model.parameters() finds all of them

    def forward(self, x):
        # Define the data flow: input -> linear -> relu -> linear -> logits
        h = self.relu(self.layer1(x))   # [batch, hidden_dim]
        return self.layer2(h)           # [batch, num_classes]

# Instantiate and move to device in ONE call
pt.manual_seed(SEED)
model = ComplaintClassifier().to(device)

# Inspect the model
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")
# 6*16 + 16 + 16*3 + 3 = 96 + 16 + 48 + 3 = 163

# Make synthetic complaint data
pt.manual_seed(SEED)
X_data = pt.randn(300, 6, dtype=pt.float32).to(device)
y_data = pt.randint(0, 3, (300,), dtype=pt.long).to(device)
dataset = TensorDataset(X_data, y_data)
loader  = DataLoader(dataset, batch_size=32, shuffle=True)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()    # combines LogSoftmax + NLL - standard for classification
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

# Training loop: 5 epochs
print("\n--- Training ComplaintClassifier (5 epochs) ---")
for epoch in range(5):
    model.train()   # enables dropout, batch norm if present (good habit)
    total_loss = 0.0
    correct = 0

    for X_batch, y_batch in loader:
        # Forward
        logits = model(X_batch)         # [batch, 3]
        loss   = criterion(logits, y_batch)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Accumulate metrics
        total_loss += loss.item() * len(y_batch)
        preds = logits.argmax(dim=1)
        correct += (preds == y_batch).sum().item()

    avg_loss = total_loss / len(dataset)
    accuracy = correct / len(dataset)
    print(f"  Epoch {epoch+1}: loss={avg_loss:.4f}  acc={accuracy:.3f}")

# Inference mode
model.eval()
with pt.no_grad():
    sample = pt.randn(1, 6, dtype=pt.float32).to(device)
    logits = model(sample)
    probs  = pt.softmax(logits, dim=1)
    pred   = logits.argmax(dim=1)
    categories = ["account_issue", "fraud", "payment"]
    print(f"\nSample prediction: {categories[pred.item()]}")
    print(f"Probabilities: {[f'{p:.3f}' for p in probs[0].tolist()]}")

### Lab 4 - Build a Deeper Complaint Classifier (Tier 1, ~20 min)

**Situation**: The two-layer classifier from Beat 3 gets about 35% accuracy on
random data (roughly chance for 3 classes). Your team wants to try a deeper network:
three hidden layers with dropout for regularization.

**Task**: Build a `DeeperComplaintClassifier` with three hidden layers (sizes 32, 16, 8)
and dropout (p=0.3) after the first two hidden layers. Train it for 10 epochs.

**Action**: Complete the steps marked `# YOUR CODE`.

**Result**: The verification cell confirms the model has more than 300 parameters
and that training loss decreases over 10 epochs.

Steps:
1. In `__init__`: create `self.layer1` (6->32), `self.drop1` (p=0.3),
   `self.layer2` (32->16), `self.drop2` (p=0.3), `self.layer3` (16->8),
   `self.output` (8->3), and `self.relu`
2. In `forward`: chain them in order (layer1 -> relu -> drop1 -> layer2 -> relu ->
   drop2 -> layer3 -> relu -> output)
3. Instantiate the model, move to `device`
4. Create `optimizer` (AdamW, lr=1e-3) and `criterion` (CrossEntropyLoss)
5. Train for 10 epochs using the existing `loader` from Beat 3
6. Record training losses in a list `deep_losses`

**Stretch**: Add a `batch_norm` layer (`nn.BatchNorm1d(32)`) between `layer1` and
`relu`. Does it train faster (fewer epochs to reach the same loss)?

In [ ]:
# Lab 4 - Deeper Complaint Classifier with nn.Module (SOLUTION)
pt.manual_seed(SEED)

class DeeperComplaintClassifier(nn.Module):
    """Three-layer classifier with dropout for Barclays complaint classification.

    Architecture: 6 -> 32 -> 16 -> 8 -> 3
    Dropout after layers 1 and 2 prevents overfitting.
    Common mistake: forgetting super().__init__() or using None for layers.
    """
    def __init__(self, input_dim=6, num_classes=3):
        super().__init__()
        # Each nn.Linear is registered as a Parameter automatically
        # PyTorch will track all of them via model.parameters()
        self.layer1 = nn.Linear(input_dim, 32)   # 6*32 + 32 = 224 params
        self.drop1  = nn.Dropout(0.3)             # randomly zeros 30% of activations
        self.layer2 = nn.Linear(32, 16)           # 32*16 + 16 = 528 params
        self.drop2  = nn.Dropout(0.3)
        self.layer3 = nn.Linear(16, 8)            # 16*8 + 8 = 136 params
        self.output = nn.Linear(8, num_classes)   # 8*3 + 3 = 27 params
        self.relu   = nn.ReLU()
        # Total: 224 + 528 + 136 + 27 = 915 params

    def forward(self, x):
        # Data flows: input -> layer1 -> relu -> drop1 -> layer2 -> relu -> drop2
        #           -> layer3 -> relu -> output logits
        # Note: dropout is active during .train() and inactive during .eval()
        x = self.drop1(self.relu(self.layer1(x)))   # [batch, 32]
        x = self.drop2(self.relu(self.layer2(x)))   # [batch, 16]
        x = self.relu(self.layer3(x))               # [batch, 8]
        return self.output(x)                        # [batch, 3] logits

# Step 3: Instantiate - .to(device) moves ALL registered parameters at once
deep_model = DeeperComplaintClassifier().to(device)

# Step 4: CrossEntropyLoss expects raw logits (not softmax output)
deep_criterion = nn.CrossEntropyLoss()
# AdamW includes weight decay by default - better than vanilla Adam for NLP
deep_optimizer = optim.AdamW(deep_model.parameters(), lr=1e-3)

# Step 5-6: Training loop - 10 epochs
deep_losses = []
for epoch in range(10):
    deep_model.train()   # important: enables dropout
    epoch_loss = 0.0
    for X_batch, y_batch in loader:
        logits = deep_model(X_batch)               # forward pass
        loss   = deep_criterion(logits, y_batch)   # compute loss
        deep_optimizer.zero_grad()                  # zero grads BEFORE backward
        loss.backward()                             # compute gradients
        deep_optimizer.step()                       # update weights
        epoch_loss += loss.item() * len(y_batch)
    deep_losses.append(epoch_loss / len(dataset))
    if (epoch + 1) % 2 == 0:
        print(f"  Epoch {epoch+1}: loss={deep_losses[-1]:.4f}")

In [ ]:
# Lab 4 safety-net: run this if you did not finish Lab 4.
# SKIP this cell if you DID finish Lab 4.
pt.manual_seed(SEED)
if deep_model is None or not deep_losses:
    print("Using Lab 4 safety-net so the rest of the notebook can run.")

    class _DeeperComplaintClassifier(nn.Module):
        def __init__(self, input_dim=6, num_classes=3):
            super().__init__()
            self.layer1 = nn.Linear(input_dim, 32)
            self.drop1  = nn.Dropout(0.3)
            self.layer2 = nn.Linear(32, 16)
            self.drop2  = nn.Dropout(0.3)
            self.layer3 = nn.Linear(16, 8)
            self.output = nn.Linear(8, num_classes)
            self.relu   = nn.ReLU()
        def forward(self, x):
            x = self.drop1(self.relu(self.layer1(x)))
            x = self.drop2(self.relu(self.layer2(x)))
            x = self.relu(self.layer3(x))
            return self.output(x)

    deep_model = _DeeperComplaintClassifier().to(device)
    deep_criterion = nn.CrossEntropyLoss()
    deep_optimizer = optim.AdamW(deep_model.parameters(), lr=1e-3)
    deep_losses = []
    for epoch in range(10):
        deep_model.train()
        epoch_loss = 0.0
        for X_batch, y_batch in loader:
            logits = deep_model(X_batch)
            loss   = deep_criterion(logits, y_batch)
            deep_optimizer.zero_grad()
            loss.backward()
            deep_optimizer.step()
            epoch_loss += loss.item() * len(y_batch)
        deep_losses.append(epoch_loss / len(dataset))

In [ ]:
# Verification - Lab 4
assert deep_model is not None, "deep_model not created"
n_params = sum(p.numel() for p in deep_model.parameters())
assert n_params > 300, f"Expected >300 params, got {n_params}"
assert len(deep_losses) == 10, "Expected 10 loss values"
assert deep_losses[-1] < deep_losses[0], "Loss should decrease"
print(f"Lab 4 passed. Parameters: {n_params}")
print(f"Loss went from {deep_losses[0]:.4f} to {deep_losses[-1]:.4f}")

**Homework Extension 4 - nn.Module**

1. Add a `save_checkpoint` method to `DeeperComplaintClassifier` that calls
   `torch.save(self.state_dict(), path)` and a `load_checkpoint` class method
   that loads it back. Verify that the loaded model produces identical predictions
   to the trained model.
2. Register a forward hook on `self.layer2` using `module.register_forward_hook(fn)`.
   Print the shape of the intermediate activation every time `forward` is called.
   Hooks are how PyTorch visualizations (like Captum) inspect model internals.
3. (Challenge) Implement `L2 regularization` manually: after `loss.backward()` but
   before `optimizer.step()`, add `lambda * param.grad.add_(param.data)` for each
   parameter. Compare this to `AdamW(weight_decay=lambda)`. Are the results identical?

## Wrap-Up - What You Built

In this notebook you built the core layers of the PyTorch stack, from raw numbers
to a working classifier:

| Component | What it does | Barclays use |
|-----------|-------------|--------------|
| Tensor | Stores and operates on numbers | Complaint feature vectors |
| Dataset + DataLoader | Batches data for efficient training | Feeds 10k complaints per hour |
| nn.Module | Defines model architecture | Custom complaint classifier |

### Key rules to remember

- Always `optimizer.zero_grad()` before `loss.backward()` - gradients accumulate.
- Move both the model and the data to the same `device`.
- Use `model.train()` and `model.eval()` to switch modes; wrap inference in `pt.no_grad()`.
- `nn.Linear` registers weights and bias as Parameters automatically.

### Optional next step

Want the internals? The companion notebook `pytorch_optional_deep_dive.ipynb`
covers the manual autograd training loop and the HuggingFace Trainer. It is
nice-to-know, not required - you are already ready for the fine-tuning topics.


In [ ]:
# Final sanity check - confirms all core sections completed successfully
checks = {
    "X_complaints (Section 1)": lambda: X_complaints.shape == (200, 6),
    "train_loader (Section 2)": lambda: train_loader is not None,
    "model (Section 3)":        lambda: isinstance(model, nn.Module),
}

all_pass = True
for name, check_fn in checks.items():
    try:
        result = check_fn()
        status = "PASS" if result else "FAIL"
        if not result:
            all_pass = False
    except Exception as e:
        status = f"ERROR: {e}"
        all_pass = False
    print(f"  {status}: {name}")

if all_pass:
    print("\nAll core sections complete. You are ready for the fine-tuning topics.")
else:
    print("\nSome sections incomplete. Re-run the safety-net cells for any FAIL/ERROR rows.")
